# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okashaahmed2/Flyrankaiintern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
# Unit of analysis(Grain):1 row = 1 unique content item (content_id) per month snapshot
# Time Window: Mid-panel observation snapshot = 2026-03 (March 2026).
#Primary Table: Flyrank internship-warehouse (Warehouse daily/monthly search performance data).
##1(need refresh) AND 0(healthy)
#Excluded: Future post-observation outcome data (metrics after March 2026) to ensure features are strictly knowable at the decision moment and avoid data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
# #Feature X : impression , ctr , avg position , word count , engagementrate ,scroll_rate (Search performance and engagement metrics used to signal page health).
# #label Y : Target variable 0 and 1{1(need refresh) AND 0(healthy)}
#Context Columns: content_id, client_id, month (Unique identifiers and
#metadata used for grouping and auditing, not as mathematical inputs).

##Excluded : we are not give the data after march 2026 to the model for leaning and its prevent data leakage

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import os
import pandas as pd
from google.colab import userdata


try:
    hf_token = userdata.get('HF_TOKEN')
    print("HF Token loaded successfully.")
except Exception as e:
    print("Please set HF_TOKEN in Colab Secrets (Key icon on left panel).")

url = "https://raw.githubusercontent.com/okashaahmed2/Flyrankaiintern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df['target'] = (df['trend_direction'] == 'down').astype(int)


# Fact 1: Grain Check (Is content_id unique?)
total_rows = len(df)
unique_content = df['content_id'].nunique()
print(f"Fact 1 - Grain Check: Total rows = {total_rows:,} | Unique content_ids = {unique_content:,}")
assert total_rows == unique_content, "Grain mismatch! Row is not unique per content_id."

# Fact 2: Row Count & Target Breakdown
print(f"\nFact 2 - Slice Row Count: {total_rows:,} rows")
print("Target Breakdown (1 = Needs Refresh, 0 = Healthy):")
print(df['target'].value_counts(normalize=True).map('{:.2%}'.format))

# Fact 3: Availability / Filter Check (IS TRUE simulation)
print(f"\nAvailable columns in DataFrame: {df.columns.tolist()}")
active_rows = df[df['impression_tier'].notnull()]
print(f"\nFact 3 - Surviving Active Rows: {len(active_rows):,} / {total_rows:,} ({len(active_rows)/total_rows:.1%})")

HF Token loaded successfully.
Fact 1 - Grain Check: Total rows = 30,000 | Unique content_ids = 30,000

Fact 2 - Slice Row Count: 30,000 rows
Target Breakdown (1 = Needs Refresh, 0 = Healthy):
target
1    54.21%
0    45.79%
Name: proportion, dtype: object

Available columns in DataFrame: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impressi

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Five Safe Features Frame
safe_features = ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count']
X_honest = df[safe_features]
y = df['target']

# Impute missing values in X_honest before fitting
X_honest = X_honest.fillna(X_honest.mean())

# 2. Honest Model Score
model = LogisticRegression(max_iter=1000)
model.fit(X_honest, y)
honest_pred = model.predict(X_honest)
honest_f1 = f1_score(y, honest_pred)

print(f"--- HONEST MODEL SCORE ---")
print(f"5 Safe Features F1-Score: {honest_f1:.4f}")

# 3. THE LEAKAGE TRAP (Adding label-derived 'trend_pct')
X_leaked = X_honest.copy()
X_leaked['trend_pct'] = df['trend_pct']  # LEAK!

# Impute missing values in X_leaked before fitting
X_leaked = X_leaked.fillna(X_leaked.mean())

model_leaked = LogisticRegression(max_iter=1000)
model_leaked.fit(X_leaked, y)
leaked_pred = model_leaked.predict(X_leaked)
leaked_f1 = f1_score(y, leaked_pred)

print(f"\n--- THE LEAKAGE TRAP EXPERIMENT ---")
print(f"Leaked Model F1-Score: {leaked_f1:.4f} (Fake Perfect Score!)")

# 4. REMOVE THE TRAP (Keeping the honest frame)
del X_leaked
print("\n[SUCCESS] Leaked column identified, demonstrated, and removed. Honest frame kept!")

--- HONEST MODEL SCORE ---
5 Safe Features F1-Score: 0.6792

--- THE LEAKAGE TRAP EXPERIMENT ---
Leaked Model F1-Score: 0.9998 (Fake Perfect Score!)

[SUCCESS] Leaked column identified, demonstrated, and removed. Honest frame kept!


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# DATA LIMIT AND LIMITATIONS
# LACK OF  CAUSAL PROOF: The dataset track observed metric drops (impression, ctr),
#but cannot prove why the drop happened (eg: Google core update vs competitors actions)

#Missing Signals: External factor signals
# (e.g., page load speeds, backlink losses, technical SEO errors) are not captured in this slice.

#Historical Depth Variance: Newer clients or low-traffic pages lack deep 90-day
# historical metrics, creating noise for edge cases.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.